# House Price Prediction - Model Training

Train machine learning models to predict Vietnamese house prices using geospatial features from Supabase.


In [2]:
# Importsimport sysfrom pathlib import Pathimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressorfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.metrics import mean_squared_error, r2_score, mean_absolute_errorimport joblibimport pickleimport warningswarnings.filterwarnings('ignore')# Add parent directory to pathsys.path.insert(0, str(Path.cwd().parent.parent))from pipeline.supabase_handler import fetch_csv_from_supabaseprint("✅ Imports successful")

✅ Imports successful


## Step 1: Load Data from Supabase


In [3]:
# Load dataprint("Loading data from Supabase...")df = fetch_csv_from_supabase("Raw_Features")print(f"✓ Loaded {len(df)} records")print(f"\nShape: {df.shape}")print(f"\nColumns: {df.columns.tolist()}")

Loading data from Supabase...
📖 Fetching data from Supabase table: alonhadat_features
❌ Error fetching from Supabase: {'message': "Could not find the table 'public.alonhadat_features' in the schema cache", 'code': 'PGRST205', 'hint': "Perhaps you meant the table 'public.Raw_Features'", 'details': None}
✓ Loaded 0 records

Shape: (0, 0)

Columns: []


In [4]:
# Inspect data
print("First 5 rows:")
df.head()

First 5 rows:


""


In [5]:
# Data info
print("\nMissing values:")
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)


Missing values:


Series([], dtype: float64)

## Step 2: Prepare Data


In [6]:
# Define features and target
FEATURE_COLS = [
    'nearest_school_km', 'school_count_3km',
    'nearest_hospital_km', 'hospital_count_5km',
    'nearest_marketplace_km', 'marketplace_count_3km',
    'nearest_supermarket_km', 'supermarket_count_3km',
    'nearest_mall_km', 'mall_count_3km',
    'nearest_bus_stop_km', 'bus_stop_count_1km',
    'nearest_metro_km', 'metro_count_5km',
    'area_m2', 'distance_to_center_km',
    'locality_square', 'locality_population_density'
]

TARGET_COL = 'price_billion_vnd'

# Check available features
available_features = [col for col in FEATURE_COLS if col in df.columns]
missing_features = [col for col in FEATURE_COLS if col not in df.columns]

print(f"✓ Available features: {len(available_features)}/{len(FEATURE_COLS)}")
if missing_features:
    print(f"⚠️  Missing: {missing_features}")

✓ Available features: 0/18
⚠️  Missing: ['nearest_school_km', 'school_count_3km', 'nearest_hospital_km', 'hospital_count_5km', 'nearest_marketplace_km', 'marketplace_count_3km', 'nearest_supermarket_km', 'supermarket_count_3km', 'nearest_mall_km', 'mall_count_3km', 'nearest_bus_stop_km', 'bus_stop_count_1km', 'nearest_metro_km', 'metro_count_5km', 'area_m2', 'distance_to_center_km', 'locality_square', 'locality_population_density']


In [7]:
# Check target column
if TARGET_COL not in df.columns:
    print(f"⚠️  Target column '{TARGET_COL}' not found")
    # Try alternative name
    if 'price' in df.columns:
        df[TARGET_COL] = df['price']
        print(f"   Using 'price' column instead")

print(f"\nTarget statistics:")
print(df[TARGET_COL].describe())

⚠️  Target column 'price_billion_vnd' not found

Target statistics:


KeyError: 'price_billion_vnd'

In [ ]:
# Prepare data
# Drop rows with missing target
df_clean = df.dropna(subset=[TARGET_COL]).copy()
print(f"After dropping NaN prices: {len(df_clean)} records")

# Select features and target
X = df_clean[available_features].copy()
y = df_clean[TARGET_COL].copy()

# Handle missing values in features
X = X.fillna(X.median())
print(f"After filling missing features: {len(X)} records")

# Remove outliers (price > 100 billion is unrealistic)
mask = y < 100
X_clean = X[mask]
y_clean = y[mask]
removed = len(X) - len(X_clean)
print(f"After removing outliers (>100B): {len(X_clean)} records (removed {removed})")

X, y = X_clean, y_clean

## Step 3: Split Data


In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nFeatures used: {len(available_features)}")
print(f"Features: {available_features}")

## Step 4: Train Models


In [ ]:
# Train Random Forest
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predict
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluate
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
rf_mae = mean_absolute_error(y_test, y_test_pred_rf)

print(f"✓ Random Forest trained")
print(f"  Train R²: {rf_train_r2:.4f}")
print(f"  Test R²: {rf_test_r2:.4f}")
print(f"  RMSE: {rf_rmse:.4f} billion VND")
print(f"  MAE: {rf_mae:.4f} billion VND")

In [ ]:
# Train Gradient Boosting
print("Training Gradient Boosting Regressor...")
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    random_state=42
)
gb_model.fit(X_train, y_train)

# Predict
y_train_pred_gb = gb_model.predict(X_train)
y_test_pred_gb = gb_model.predict(X_test)

# Evaluate
gb_train_r2 = r2_score(y_train, y_train_pred_gb)
gb_test_r2 = r2_score(y_test, y_test_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_gb))
gb_mae = mean_absolute_error(y_test, y_test_pred_gb)

print(f"✓ Gradient Boosting trained")
print(f"  Train R²: {gb_train_r2:.4f}")
print(f"  Test R²: {gb_test_r2:.4f}")
print(f"  RMSE: {gb_rmse:.4f} billion VND")
print(f"  MAE: {gb_mae:.4f} billion VND")

## Step 5: Compare Models


In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting'],
    'Train R²': [rf_train_r2, gb_train_r2],
    'Test R²': [rf_test_r2, gb_test_r2],
    'RMSE': [rf_rmse, gb_rmse],
    'MAE': [rf_mae, gb_mae]
})

print("\nModel Comparison:")
print(comparison.to_string(index=False))

# Select best
best_model_name = 'Random Forest' if rf_test_r2 > gb_test_r2 else 'Gradient Boosting'
best_model = rf_model if rf_test_r2 > gb_test_r2 else gb_model
best_r2 = max(rf_test_r2, gb_test_r2)

print(f"\n✅ Best Model: {best_model_name} (R² = {best_r2:.4f})")

## Step 6: Feature Importance


In [ ]:
# Get feature importance from best model
if isinstance(best_model, RandomForestRegressor):
    importances = best_model.feature_importances_
elif isinstance(best_model, GradientBoostingRegressor):
    importances = best_model.feature_importances_

# Sort by importance
feature_importance = pd.DataFrame({
    'Feature': available_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'].head(10), feature_importance['Importance'].head(10))
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.tight_layout()
plt.show()

## Step 7: Prediction Analysis


In [ ]:
# Analyze predictions
y_test_pred = best_model.predict(X_test)
errors = y_test - y_test_pred

print("\nPrediction Error Statistics:")
print(f"  Mean Error: {errors.mean():.4f} billion VND")
print(f"  Std Dev: {errors.std():.4f} billion VND")
print(f"  Min Error: {errors.min():.4f} billion VND")
print(f"  Max Error: {errors.max():.4f} billion VND")
print(f"  Median Error: {np.median(errors):.4f} billion VND")

# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_test_pred, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price (billion VND)')
axes[0].set_ylabel('Predicted Price (billion VND)')
axes[0].set_title('Actual vs Predicted Prices')
axes[0].grid(True, alpha=0.3)

# Error distribution
axes[1].hist(errors, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Prediction Error (billion VND)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prediction Errors')
axes[1].axvline(errors.mean(), color='r', linestyle='--', label=f'Mean: {errors.mean():.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Save Model


In [ ]:
# Create saved_models directory
model_dir = Path('saved_models')
model_dir.mkdir(exist_ok=True)

# Save model
model_file = best_model_name.lower().replace(' ', '_')
model_path = model_dir / f"{model_file}.joblib"
joblib.dump(best_model, model_path)
print(f"✓ Saved model: {model_path}")

# Save metadata
metadata = {
    'model_type': best_model_name,
    'features': available_features,
    'metrics': {
        'train_r2': float(rf_train_r2 if best_model_name == 'Random Forest' else gb_train_r2),
        'test_r2': float(best_r2),
        'rmse': float(np.sqrt(mean_squared_error(y_test, y_test_pred))),
        'mae': float(mean_absolute_error(y_test, y_test_pred))
    },
    'train_size': len(X_train),
    'test_size': len(X_test)
}

meta_path = model_dir / f"{model_file}_meta.pkl"
with open(meta_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✓ Saved metadata: {meta_path}")

print(f"\n✅ Model saved successfully!")
print(f"   Next: Use models/inference/predict.ipynb to make predictions")